# Getting Started: The Execution Edge Challenge

This notebook provides a step-by-step guide to get started with the project.

## Step 1: Data Exploration
First, let's understand the data structure.


In [87]:
import polars as pl
import numpy as np
import pandas as pd
import re
from pathlib import Path
import matplotlib.pyplot as plt

# Set data directory
DATA_DIR = "BTCUSDT"  # Start with BTCUSDT

# Load data
X_train = pl.read_parquet(f"{DATA_DIR}/X_train.parquet")
y_train = pl.read_parquet(f"{DATA_DIR}/y_train.parquet")

# Load volume to fill
with open(f"{DATA_DIR}/vol_to_fill.txt", "r") as file:
    content = file.read().strip()
match = re.search(r"The volume to fill is: ([\d.]+)", content)
volume_to_fill = float(match.group(1)) if match else None

print(f"Volume to fill: {volume_to_fill}")
print(f"\nX_train shape: {X_train.shape}")
print(f"X_train columns: {X_train.columns}")
print(f"\ny_train shape: {y_train.shape}")
print(f"y_train columns: {y_train.columns}")


Volume to fill: 4.0

X_train shape: (2495722, 27)
X_train columns: ['ask_price_1', 'ask_vol_1', 'bid_price_1', 'bid_vol_1', 'ask_price_2', 'ask_vol_2', 'bid_price_2', 'bid_vol_2', 'ask_price_3', 'ask_vol_3', 'bid_price_3', 'bid_vol_3', 'ask_price_4', 'ask_vol_4', 'bid_price_4', 'bid_vol_4', 'ask_price_5', 'ask_vol_5', 'bid_price_5', 'bid_vol_5', 'open', 'high', 'low', 'close', 'volume', 'anonymized_id', 'time_in_hour']

y_train shape: (42300, 27)
y_train columns: ['ask_price_1', 'ask_vol_1', 'bid_price_1', 'bid_vol_1', 'ask_price_2', 'ask_vol_2', 'bid_price_2', 'bid_vol_2', 'ask_price_3', 'ask_vol_3', 'bid_price_3', 'bid_vol_3', 'ask_price_4', 'ask_vol_4', 'bid_price_4', 'bid_vol_4', 'ask_price_5', 'ask_vol_5', 'bid_price_5', 'bid_vol_5', 'open', 'high', 'low', 'close', 'volume', 'anonymized_id', 'time_in_hour']


In [88]:
# Explore X_train structure
print("X_train sample:")
print(X_train.head())
print("\n" + "="*50)
print("\nData types:")
print(X_train.dtypes)
print("\n" + "="*50)
print("\nUnique anonymized_ids:")
print(X_train['anonymized_id'].n_unique())
print("\nUnique time_in_hour values:")
print(X_train['time_in_hour'].unique().sort()[:10])


X_train sample:
shape: (5, 27)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬──────────┬───────────┬───────────┐
│ ask_price ┆ ask_vol_1 ┆ bid_price ┆ bid_vol_1 ┆ … ┆ close     ┆ volume   ┆ anonymize ┆ time_in_h │
│ _1        ┆ ---       ┆ _1        ┆ ---       ┆   ┆ ---       ┆ ---      ┆ d_id      ┆ our       │
│ ---       ┆ f64       ┆ ---       ┆ f64       ┆   ┆ f64       ┆ f64      ┆ ---       ┆ ---       │
│ f64       ┆           ┆ f64       ┆           ┆   ┆           ┆          ┆ u64       ┆ duration[ │
│           ┆           ┆           ┆           ┆   ┆           ┆          ┆           ┆ ms]       │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪══════════╪═══════════╪═══════════╡
│ 56892.528 ┆ 0.649013  ┆ 56892.474 ┆ 0.520792  ┆ … ┆ null      ┆ null     ┆ 344207310 ┆ 0ms       │
│           ┆           ┆ 1         ┆           ┆   ┆           ┆          ┆ 498719334 ┆           │
│           ┆           ┆           ┆           ┆   ┆       

In [89]:
X_train.head()

ask_price_1,ask_vol_1,bid_price_1,bid_vol_1,ask_price_2,ask_vol_2,bid_price_2,bid_vol_2,ask_price_3,ask_vol_3,bid_price_3,bid_vol_3,ask_price_4,ask_vol_4,bid_price_4,bid_vol_4,ask_price_5,ask_vol_5,bid_price_5,bid_vol_5,open,high,low,close,volume,anonymized_id,time_in_hour
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u64,duration[ms]
56892.528,0.649013,56892.4741,0.520792,56892.5819,0.00234,56892.0968,0.001528,56894.0911,0.004734,56891.9351,0.000405,56895.1152,0.00374,56891.3961,0.03,56895.2769,0.06988,56889.7252,0.050882,null,null,null,null,null,3442073104987193344,0ms
56892.528,0.7094365,56877.7055,0.350604,56900.1279,0.097365,56875.73815,0.00071,56900.491725,0.080226,56884.6047,0.042489,56904.183875,0.081072,56882.79905,0.032681,56903.6853,0.001578,56879.1069,0.0,56892.528,56892.528,56892.528,56892.528,0.001545,3442073104987193344,1s
56894.28514,0.567232,56892.4741,0.768055,56900.96874,0.020947,56877.579733,0.071667,56904.41295,0.081071,56869.279133,0.245209,56904.5477,0.022358,56871.4531,0.0,56913.4412,0.164116,56858.571,0.020296,null,null,null,null,null,3442073104987193344,2s
56892.528,0.7090038,56891.00802,0.6482008,56898.79118,0.152273,56884.083667,0.388231,56901.86348,0.2499924,56882.089367,0.065192,56905.17294,0.078377,56880.1849,0.08308,56907.4583,0.298564,56875.0105,0.076982,56892.528,56892.528,56892.528,56892.528,0.000047,3442073104987193344,3s
56892.528,0.702552,56887.17034,0.676458,56899.521525,0.1019585,56866.831175,0.000756,56899.99315,0.0194685,56863.354625,0.003211,56899.517033,0.001041,56866.09005,0.0073815,56899.768567,0.000947,56873.2318,0.001209,56892.528,56892.528,56892.528,56892.528,0.00811,3442073104987193344,4s


In [90]:
y_train[[
"close", "open"
]]

close,open
f64,f64
null,null
null,null
56841.862,56841.862
56841.9159,56841.9159
null,null
…,…
138806.5732,138806.4305
138806.4305,138806.4305
138806.4305,138806.4305


## Step 2: Set Up Evaluation Function

We need to implement the scoring function that simulates walking the order book.


In [91]:
# Import the simulate_walk_the_book function
import sys
sys.path.append('.')
from simulate_walk_the_book import simulate_walk_the_book

# Helper function to compute implementation error
def compute_implementation_error(
    y_submission: pl.DataFrame,
    y_train: pl.DataFrame,
    volume_to_fill: float
) -> float:
    """
    Compute implementation error for a submission.
    
    Args:
        y_submission: DataFrame with columns [anonymized_id, time_in_hour, position]
                     Contains positions for the last 60 seconds of each hour
        y_train: DataFrame with order book data (ask/bid prices/volumes) and close prices
                 for the last minute of each hour.
        volume_to_fill: Target volume to execute
    
    Returns:
        Mean implementation error in basis points
    
    Formula:
        error = |avg_price - close_price| / close_price × min(100, volume_to_fill / total_vol) × 10000
    """
    # Join submission with y_train to get order book data for each position
    # y_train contains: order book data (ask/bid prices/volumes) + close prices
    merged = (
        y_submission
        .join(y_train, on=['anonymized_id', 'time_in_hour'], how='left')
        .sort(['anonymized_id', 'time_in_hour'])
    )
    
    errors = []
    
    for anonymized_id in merged['anonymized_id'].unique():
        id_data = merged.filter(pl.col('anonymized_id') == anonymized_id)
        
        # Get close price from y_train - the LAST close price in the last minute
        # Formula from example.ipynb: y_last_min.select("close").drop_nulls().to_series().last()
        # This is the close price at the END of the hour (last second of last minute)
        close_prices = id_data['close'].drop_nulls()
        if len(close_prices) == 0:
            continue
        close_price = close_prices[-1]  # Get the LAST close price (end of hour)
        
        # Prepare order book arrays
        positions = id_data['position'].fill_null(0.0).to_numpy()
        
        # Extract ask/bid prices and volumes (5 levels)
        ask_prices = np.column_stack([
            id_data[f'ask_price_{i}'].fill_null(np.nan).to_numpy() 
            for i in range(1, 6)
        ])
        ask_vols = np.column_stack([
            id_data[f'ask_vol_{i}'].fill_null(np.nan).to_numpy() 
            for i in range(1, 6)
        ])
        bid_prices = np.column_stack([
            id_data[f'bid_price_{i}'].fill_null(np.nan).to_numpy() 
            for i in range(1, 6)
        ])
        bid_vols = np.column_stack([
            id_data[f'bid_vol_{i}'].fill_null(np.nan).to_numpy() 
            for i in range(1, 6)
        ])
        
        # Simulate walk the book
        total_vol, avg_price = simulate_walk_the_book(
            positions, ask_prices, ask_vols, bid_prices, bid_vols
        )
        
        if total_vol == 0 or np.isnan(avg_price) or np.isnan(close_price):
            continue
        
        # Compute implementation error
        relative_error = abs(avg_price - close_price) / close_price
        fill_penalty = min(100, volume_to_fill / total_vol) if total_vol > 0 else 100
        
        error_bps = relative_error * fill_penalty * 10000  # Convert to basis points
        errors.append(error_bps)
    
    return np.mean(errors) if errors else np.nan


## Step 3: Create Baseline Models

Let's start with simple baselines to establish a benchmark.


## Step 3: Evaluate Baseline Models

Let's evaluate both baselines to see which performs better!


In [92]:
# Helper function to create submission template
def create_submission_template(y_train: pl.DataFrame) -> pl.DataFrame:
    """Create a template with all anonymized_ids and time_in_hour combinations."""
    all_time_in_hour = y_train['time_in_hour'].unique().sort()
    unique_ids = y_train['anonymized_id'].unique().sort()
    
    templated_df = unique_ids.to_frame().join(
        all_time_in_hour.to_frame(),
        how="cross"
    )
    
    return templated_df

# Baseline 1: Uniform distribution over all 60 seconds
def baseline_uniform(y_train: pl.DataFrame, volume_to_fill: float) -> pl.DataFrame:
    """Distribute volume uniformly across all seconds."""
    template = create_submission_template(y_train)
    
    submission = template.with_columns(
        position=pl.lit(volume_to_fill / 60)
    )
    
    return submission

# Baseline 2: Fill only at the end of the hour
def baseline_end_of_hour(y_train: pl.DataFrame, volume_to_fill: float, 
                        seconds_from_end: int = 14) -> pl.DataFrame:
    """Fill volume only in the last N seconds of the hour."""
    template = create_submission_template(y_train)
    
    # Get the last time_in_hour value
    last_time = y_train['time_in_hour'].max()
    
    # Create threshold (last_time - seconds_from_end seconds)
    threshold = last_time - pl.duration(seconds=seconds_from_end)
    
    submission = template.with_columns(
        position=pl.when(pl.col('time_in_hour') > threshold)
        .then(pl.lit(volume_to_fill / seconds_from_end))
        .otherwise(pl.lit(0.0))
    )
    
    return submission

# Test baselines
template = create_submission_template(y_train)
print(f"Template shape: {template.shape}")
print(f"Expected: {y_train['anonymized_id'].n_unique()} IDs × 60 seconds")

baseline1 = baseline_uniform(y_train, volume_to_fill)
print(f"\nBaseline 1 (uniform) - Total position: {baseline1['position'].sum()}")

baseline2 = baseline_end_of_hour(y_train, volume_to_fill, seconds_from_end=14)
print(f"Baseline 2 (end of hour) - Total position: {baseline2['position'].sum()}")


Template shape: (42300, 2)
Expected: 705 IDs × 60 seconds

Baseline 1 (uniform) - Total position: 2820.0
Baseline 2 (end of hour) - Total position: 2820.0


In [93]:
# Evaluate both baselines
print("Evaluating baselines...")
print("This may take a few minutes...\n")

# Evaluate baseline 1 (uniform)
error1 = compute_implementation_error(baseline1, y_train, volume_to_fill)
print(f"Baseline 1 (uniform): {error1:.4f} bps")

# Evaluate baseline 2 (end of hour, 14 seconds)
error2 = compute_implementation_error(baseline2, y_train, volume_to_fill)
print(f"Baseline 2 (end of hour, 14s): {error2:.4f} bps")

# Compare with spread benchmark
spread = 20000 * (y_train['ask_price_1'] - y_train['bid_price_1']) / (
    y_train['ask_price_1'] + y_train['bid_price_1']
)
spread_mean = spread.mean()
print(f"\nSpread benchmark: {spread_mean:.4f} bps")

# Summary
print("\n" + "="*50)
print("SUMMARY:")
print("="*50)
if error1 < error2:
    print(f"✓ Baseline 1 (uniform) is better: {error1:.4f} bps")
else:
    print(f"✓ Baseline 2 (end of hour) is better: {error2:.4f} bps")

if min(error1, error2) < spread_mean:
    print(f"✓ Best baseline beats spread benchmark!")
else:
    print(f"✗ Best baseline does NOT beat spread benchmark")
    print(f"  Need to improve by at least {min(error1, error2) - spread_mean:.4f} bps")


Evaluating baselines...
This may take a few minutes...

Baseline 1 (uniform): 1.5121 bps
Baseline 2 (end of hour, 14s): 1.2186 bps

Spread benchmark: 1.4197 bps

SUMMARY:
✓ Baseline 2 (end of hour) is better: 1.2186 bps
✓ Best baseline beats spread benchmark!


## Step 4: Advanced Strategy - Threshold-Based Execution

Let's build a more sophisticated strategy using realistic trading logic.


**The Real Problem**: We're optimizing the WRONG thing!

The metric is `|average_price - close_price|`, NOT execution cost!
- End-of-hour works because prices near the end are closer to close_price
- Our threshold strategy optimizes spread/liquidity, but that doesn't help match close_price

**Better Approach**: Since we can't know close_price ahead of time, we should:
1. **Trade more at the end** (prices converge to close_price)
2. **Within that window, use current liquidity** - but make decisions sequentially!
3. **NO future information** - at each second, decide based on:
   - Current liquidity (we know this NOW)
   - Remaining volume
   - Remaining time

This is realistic and doesn't peek ahead!


In [94]:
def baseline_smart_end_of_hour(
    y_train: pl.DataFrame,
    volume_to_fill: float,
    seconds_from_end: int = 14,
    min_liquidity_multiplier: float = 0.5,
    max_liquidity_multiplier: float = 1.5,
) -> pl.DataFrame:
    """Sequential end-of-hour strategy that scales fills based on liquidity deltas."""
    template = create_submission_template(y_train)
    merged = (
        template.join(y_train, on=['anonymized_id', 'time_in_hour'], how='left')
        .sort(['anonymized_id', 'time_in_hour'])
    )

    last_time = y_train['time_in_hour'].max()
    window_start = last_time - pl.duration(seconds=seconds_from_end) # 60s - 14s = 46s => window starts at 46s

    features = merged.with_columns([
        (pl.col('ask_vol_1') + pl.col('bid_vol_1')).fill_nan(0.0).fill_null(0.0).alias('best_liquidity'),
        (pl.col('time_in_hour') > window_start).alias('in_window'),
    ])

    def allocate_positions(group: pl.DataFrame) -> pl.DataFrame:
        ordered = group.sort('time_in_hour')
        rows = list(ordered.iter_rows(named=True))
        in_window_flags = [row['in_window'] for row in rows]

        positions = []
        prev_liquidity = None
        window_rows_seen = 0
        remaining_volume = volume_to_fill

        for idx, row in enumerate(rows):
            current_liquidity = float(row['best_liquidity'] or 0.0)
            in_window = in_window_flags[idx]

            if not in_window or remaining_volume <= 0:
                positions.append(0.0)
                continue

            remaining_slots = max(seconds_from_end - window_rows_seen, 1)
            base_allocation = remaining_volume / remaining_slots

            if prev_liquidity is None:
                liquidity_ratio = 1.0
            elif prev_liquidity == 0:
                liquidity_ratio = max_liquidity_multiplier if current_liquidity > 0 else min_liquidity_multiplier
            else:
                liquidity_ratio = current_liquidity / prev_liquidity

            liquidity_multiplier = max(
                min_liquidity_multiplier,
                min(max_liquidity_multiplier, liquidity_ratio),
            )

            position = min(base_allocation * liquidity_multiplier, remaining_volume)
            position = max(0.0, position)

            positions.append(position)
            remaining_volume -= position
            window_rows_seen += 1
            prev_liquidity = current_liquidity

        if window_rows_seen == 0:
            positions = [0.0 for _ in positions]

        return ordered.with_columns(pl.Series('position', positions))

    result = features.group_by('anonymized_id').map_groups(allocate_positions)
    return result.select(['anonymized_id', 'time_in_hour', 'position'])

## Step 5: Evaluate Threshold-Based Strategy

Let's test the threshold-based strategy and compare it with our baselines.


In [95]:
# Calculate spread benchmark
spread = 20000 * (y_train['ask_price_1'] - y_train['bid_price_1']) / (
    y_train['ask_price_1'] + y_train['bid_price_1']
)
spread_mean = spread.mean()

# Test the smart end-of-hour strategy
print("Creating smart end-of-hour strategy...")
baseline3 = baseline_smart_end_of_hour(y_train, volume_to_fill)
print(f"Baseline 3 (smart end-of-hour) - Total position: {baseline3['position'].sum():.2f}")

# Evaluate it
print("\nEvaluating smart end-of-hour strategy...")
print("This may take a few minutes...")
error3 = compute_implementation_error(baseline3, y_train, volume_to_fill)
print(f"Baseline 3 (smart end-of-hour): {error3:.4f} bps")

# Compare with previous baselines
print("\n" + "="*50)
print("COMPARISON:")
print("="*50)
print(f"Baseline 1 (uniform): {error1:.4f} bps")
print(f"Baseline 2 (end of hour): {error2:.4f} bps")
print(f"Baseline 3 (smart end-of-hour): {error3:.4f} bps")
print(f"Spread benchmark: {spread_mean:.4f} bps")

# Find best
best_error = min(error1, error2, error3)
if best_error == error1:
    best_name = "uniform"
elif best_error == error2:
    best_name = "end of hour"
else:
    best_name = "smart end-of-hour"

print(f"\n✓ Best strategy so far: {best_name} with {best_error:.4f} bps")

if best_error < spread_mean:
    print(f"✓ Beats spread benchmark by {spread_mean - best_error:.4f} bps!")
else:
    print(f"✗ Still {best_error - spread_mean:.4f} bps away from spread benchmark")

print(f"\nSmart end-of-hour approach:")
print(f"  - Trades in last 14 seconds (like baseline)")
print(f"  - Makes decisions sequentially (no future information!)")
print(f"  - At each second: trade more if current liquidity is high (relative to past)")
print(f"  - Realistic: only uses information available at that moment")


Creating smart end-of-hour strategy...
Baseline 3 (smart end-of-hour) - Total position: 2797.33

Evaluating smart end-of-hour strategy...
This may take a few minutes...
Baseline 3 (smart end-of-hour): 1.2011 bps

COMPARISON:
Baseline 1 (uniform): 1.5121 bps
Baseline 2 (end of hour): 1.2186 bps
Baseline 3 (smart end-of-hour): 1.2011 bps
Spread benchmark: 1.4197 bps

✓ Best strategy so far: smart end-of-hour with 1.2011 bps
✓ Beats spread benchmark by 0.2187 bps!

Smart end-of-hour approach:
  - Trades in last 14 seconds (like baseline)
  - Makes decisions sequentially (no future information!)
  - At each second: trade more if current liquidity is high (relative to past)
  - Realistic: only uses information available at that moment


## Step 6: Next Steps

1. **Optimize thresholds**: Try different threshold values (median, mean, percentiles)
2. **Add more features**: Order book depth, price momentum, volatility
3. **Machine learning**: Train models to predict optimal positions
4. **Cross-validation**: Split data properly to avoid overfitting
5. **Test on other pairs**: Try ADAUSDT, ETHUSDT, etc.

### Key Insights to Explore:
- When is liquidity highest?
- How does spread vary throughout the hour?
- What's the relationship between order book depth and execution cost?
- Can we predict the close price from early order book data?
